# Session 19 Lab: Production Deployment
**Course:** Noob to AI Expert | **Track:** Expert

Add caching, observability logging, exponential backoff retries, and user rate limiting to your AI app.

In [ ]:
!pip install anthropic -q
print('✅ Ready')

In [ ]:
import anthropic, hashlib, json, time, logging, uuid, random, statistics
from dataclasses import dataclass, asdict
from collections import defaultdict

os_import = __import__('os')
os_import.environ.setdefault('ANTHROPIC_API_KEY', 'your-key-here')
client = anthropic.Anthropic()

# --- RESPONSE CACHE ---
class ResponseCache:
    def __init__(self): self._store = {}
    def _key(self, model, messages, system):
        return hashlib.sha256(json.dumps({'m': model, 'msg': messages, 's': system}, sort_keys=True).encode()).hexdigest()
    def get(self, model, messages, system=''):
        return self._store.get(self._key(model, messages, system))
    def set(self, model, messages, system, response):
        self._store[self._key(model, messages, system)] = response
    def stats(self): return {'size': len(self._store)}

cache = ResponseCache()

# --- OBSERVABILITY ---
@dataclass
class CallLog:
    request_id: str; model: str; input_tokens: int; output_tokens: int
    latency_ms: float; cache_hit: bool; error: str | None

logs = []

def observed_call(model, messages, system='', max_tokens=256):
    rid = str(uuid.uuid4())[:8]
    cached = cache.get(model, messages, system)
    if cached:
        logs.append(CallLog(rid, model, 0, 0, 0.0, True, None))
        return cached
    start = time.time()
    try:
        resp = client.messages.create(model=model, max_tokens=max_tokens, system=system, messages=messages)
        result = resp.content[0].text
        latency = (time.time()-start)*1000
        logs.append(CallLog(rid, model, resp.usage.input_tokens, resp.usage.output_tokens, latency, False, None))
        cache.set(model, messages, system, result)
        return result
    except Exception as e:
        logs.append(CallLog(rid, model, 0, 0, (time.time()-start)*1000, False, str(e)))
        raise

# --- RATE LIMITER ---
class RateLimiter:
    def __init__(self, rpm=5): self.limit = rpm; self.windows = defaultdict(list)
    def is_allowed(self, uid):
        now = time.time()
        self.windows[uid] = [t for t in self.windows[uid] if now-t < 60]
        if len(self.windows[uid]) >= self.limit: return False
        self.windows[uid].append(now); return True

limiter = RateLimiter(rpm=5)
print('✅ All production components initialized')

In [ ]:
# Test caching
msg = [{'role': 'user', 'content': 'What is machine learning?'}]

for i in range(3):
    start = time.time()
    result = observed_call('claude-haiku-4-5-20251001', msg)
    elapsed = (time.time()-start)*1000
    hit = logs[-1].cache_hit
    print(f'Call {i+1}: {elapsed:.1f}ms | cache={hit}')

print(f'\nCache stats: {cache.stats()}')
print(f'Total logs: {len(logs)}')
print(f'Avg latency (non-cached): {statistics.mean(l.latency_ms for l in logs if not l.cache_hit):.1f}ms')

In [ ]:
# Test rate limiting: 3 users × 6 requests
for user in ['alice', 'bob', 'charlie']:
    allowed = blocked = 0
    for i in range(6):
        if limiter.is_allowed(user): allowed += 1
        else: blocked += 1
    print(f'{user:10s}: {allowed} allowed, {blocked} blocked (limit=5/min)')